In [1]:
# import necessary libraries
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.neighbors import NearestNeighbors
from sklearn.neighbors import NearestNeighbors

**For questions 1 and 2:**

Do a regression to estimate the fixed effect of each group. We assume that there is one single linear coefficient for all the data, plus the fixed effect of each group. Use the file homework_2.1.csv.  
The variables G1, G2, and G3 are the outcomes and the time is the treatment.

## Question 1

Which of these is closest to being the coefficient of group 1? 

- A: 0.00485

- B: 0.00850

- C: 0.01023   

- D: 0.1823 

In [2]:
# load the data
df1= pd.read_csv('./data/homework_2.1.csv')
df1.head()

,time,G1,G2,G3
0,0,0.882026,1.441575,0.065409
1,1,0.210079,-0.163880,0.140310
2,2,0.509369,-0.115242,0.819830
3,3,1.150447,1.014698,0.607632
4,4,0.973779,-0.046562,0.610066


In [3]:
model_g1 = smf.ols('G1 ~ time', data=df1).fit()
print("Group 1 Slope:", model_g1.params)

Group 1 Slope: Intercept    0.104236
time         0.008498
dtype: float64


In [4]:
X = sm.add_constant(df1['time'])
model = sm.OLS(df1['G1'], X).fit()
model_g1.params

Intercept    0.104236
time         0.008498
dtype: float64

Answer : B
___

## Question 2

Which of these is closest to being the common linear coefficient for all groups?

- A: 0.01120

- B: 0.01003

- C: 0.009017  

- D: 0.2808


In [5]:
df_long = pd.melt(df1, id_vars=['time'], value_vars=['G1', 'G2', 'G3'], var_name='group', value_name='y')
df_long

,time,group,y
0,0,G1,0.882026
1,1,G1,0.210079
2,2,G1,0.509369
3,3,G1,1.150447
4,4,G1,0.973779
...,...,...,...
295,95,G3,1.768446
296,96,G3,1.258862
297,97,G3,1.511477
298,98,G3,1.030275


In [6]:
dummies = pd.get_dummies(df_long['group'], dtype=int)

X = pd.concat([df_long[['time']], dummies], axis=1)
y = df_long['y']

model = sm.OLS(y, X).fit()

model.params

time    0.009017
G1      0.078552
G2      0.589654
G3      0.269032
dtype: float64

In [7]:
model_panel = smf.ols('y ~ time + C(group) - 1', data=df_long).fit()
print("Common Time Coefficient:", model_panel.params)

Common Time Coefficient: C(group)[G1]    0.078552
C(group)[G2]    0.589654
C(group)[G3]    0.269032
time            0.009017
dtype: float64


Answer : C
___

**For questions 3-5:**

Given a data set, create a bootstrap simulation to try different possibilities.    
Use the file homework_2.2.csv 

## Question 3
If we were to measure the effect of the treatment simply by subtracting the outcomes of the treated vs. untreated population, which of these is closest to the mean effect? (This is not the recommended way of measuring the mean effect when there are confounders!) 

- A: 12.831 

- B: 2.921

- C: 5.149  

- D: 8.031


In [8]:
df2 = pd.read_csv('./data/homework_2.2.csv')
df2.head()

,X,Y,Z
0,0,1.182435,-0.725820
1,0,2.714474,0.563476
2,0,0.077612,-0.435632
3,0,-0.154449,-0.104553
4,0,22.298992,-2.321273


In [10]:
treated = df2[df2["X"] == 1]
untreated = df2[df2["X"] == 0]
treated_outcome = treated["Y"].values
untreated_outcome = untreated["Y"].values
print(f"Mean outcome for treated group: {treated['Y'].mean()}")
print(f"Mean outcome for untreated group: {untreated['Y'].mean()}")
print(f"Average treatment effect: {treated['Y'].mean() - untreated['Y'].mean()}")

Mean outcome for treated group: 7.842756767343214
Mean outcome for untreated group: 4.922039502620024
Average treatment effect: 2.9207172647231907


Answer : B 
___

## Question 4

If we were to use bootstrap sampling to measure the variance of that effect, again finding the effect using the non-recommended approach, which of these is closest to that variance?

- A: 0.001822 

- B: 0.002388  

- C: 0.03274   

- D: 0.1280


In [26]:
# make bootstrap samples
n_bootstrap = 1000
bootstrap_effects = []

for i in range(n_bootstrap):
    # sample with replacement from each group
    df_bootstrap = df2.sample(n=len(df2), replace=True)
    treated_sample = df_bootstrap[df_bootstrap["X"] == 1]
    untreated_sample = df_bootstrap[df_bootstrap["X"] == 0]

    # calculate the treatment effect for this bootstrap sample
    bootstrap_effect = (treated_sample["Y"].mean() - untreated_sample["Y"].mean())
    bootstrap_effects.append(bootstrap_effect)

print(f"Mean of bootstrap treatment effects: {np.mean(bootstrap_effects)}")
print(f"Standard deviation of bootstrap treatment effects: {np.std(bootstrap_effects)}")
print(f"variance of bootstrap treatment effects: {np.var(bootstrap_effects)}")
print(f"95% confidence interval for treatment effect: {np.percentile(bootstrap_effects, [2.5, 97.5])}")

Mean of bootstrap treatment effects: 2.929668219173187
Standard deviation of bootstrap treatment effects: 0.17840276799231547
variance of bootstrap treatment effects: 0.031827547627319944
95% confidence interval for treatment effect: [2.57873903 3.27764052]


Answer : C
___

# Question 5  

if we ran a linear regression (with intercept) to measure the effect, which of these is closest to the skewness of the effect measured? (Look up skewness online. You can use scipy.stats.skew to compute the skewness of a list of numbers.) 

- A: 0.3223

- B: 0.09812

- C: 0.04850  

- D: 0.2315


In [30]:
from scipy.stats import skew

regression_effects = []

for i in range(100000):
    df_bootstrap = df2.sample(n=len(df2), replace=True)

    model = smf.ols('Y ~ X', data=df_bootstrap).fit()

    regression_effects.append(model.params['X'])

print("Skewness:", skew(regression_effects))

Skewness: 0.04377193848432023


In [31]:
regression_effects = []
np.random.seed(42)

for i in range(10000):
    df_bootstrap = df2.sample(n=len(df2), replace=True)

    model = smf.ols('Y ~ X', data=df_bootstrap).fit()

    regression_effects.append(model.params['X'])

print("Skewness:", skew(regression_effects))

Skewness: 0.04074594584440536


Answer : C
___